- original point cloud
- fps sampled points
- random sampling comparison with fps

In [ ]:
import trimesh

import torch

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from barebones import fps

In [ ]:
mesh = trimesh.load("3d-models\drone.glb")

points, _ = trimesh.sample.sample_surface(mesh, 5000)

pts = torch.tensor(points, dtype=torch.float32)

In [ ]:
idx, sampled = fps.fps(pts, 512)

In [ ]:
# distance to nearest sampled point
dists = torch.cdist(pts, sampled) # [N, K]
min_dist, _ = dists.min(dim=1) # [N]

# min max normalization for heatmap in range [0, 1]
heat = (min_dist - min_dist.min()) / (min_dist.max() - min_dist.min())
heat_np = heat.numpy()

In [ ]:
pts_np = pts.numpy()
sampled_np = sampled.numpy()

fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(111, projection='3d')

sc = ax.scatter(
    pts_np[:,0], pts_np[:,1], pts_np[:,2],
    c=heat_np,
    s=5,
    cmap='inferno'
)
plt.colorbar(sc, label="Coverage Error")
ax.scatter(sampled_np[:,0], sampled_np[:,1], sampled_np[:,2], s=20, c='cyan')